# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RohanNK86/ML_Intern_Assets/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from huggingface_hub import login
from datasets import load_dataset
import pandas as pd

# Login using your HF Token
login(token="HF_TOKEN")

In [4]:
# Load a specific subset, e.g., 'dim_clients'
dataset = load_dataset("FlyRank/internship-warehouse", "dim_clients")

# Convert to Pandas DataFrame for analysis
df_clients = dataset['train'].to_pandas()

# Preview data
df_clients.head()

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,None,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,None,None
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,None
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,None


In [6]:
# Check distribution of access profiles across clients
print(df_clients['access_profile'].value_counts())

# Filter active clients with GSC and GA4 access
active_full_access = df_clients[
    (df_clients['is_active'] == True) &
    (df_clients['access_profile'] == 'gsc_and_ga4')
]
print(f"Active clients with full access: {len(active_full_access)}")

access_profile
gsc_and_ga4                             53
no_search_or_analytics_access           26
gsc_only                                14
source_only_missing_client_dimension    10
ga4_only                                 1
Name: count, dtype: int64
Active clients with full access: 41


In [10]:
df_clients.shape

(104, 9)

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Soln** :
* **Unit of Analysis (Grain):** One row represents a single page/content item (`content_hash_id`) for a specific client (`client_hash_id`) on a single date (`date`).
* **Tables Used:** `fact_content_daily_performance` joined with `dim_clients` and `dim_content`.
* **Time Window:** Mid-panel month of **March 2026** (`2026-03-01` to `2026-03-31`). (June 2026 is reserved as the sealed test set).
* **Target / Proxy Label:** Predicting whether a content page will generate high organic traffic in the subsequent 30-day window (`clicks_next_30d > 10`).
* **Deliberate Exclusion:** Excluded `conversions` and future-dated traffic metrics from the observation window to prevent target leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Soln** :
* **Features (Knowable prior to prediction):**
  * `clicks_7d_sum`: Trailing 7-day organic clicks.
  * `impressions_7d_sum`: Trailing 7-day search impressions.
  * `ctr_7d_avg`: Trailing 7-day average click-through rate.
  * `position_7d_avg`: Trailing 7-day average search position rank.
  * `days_since_created`: Content age in days from creation to observation date.

* **Target / Label:**
  * `is_high_performer`: Binary flag indicating if total clicks in the next 30 days exceed 10.

* **Context / Metadata:**
  * `client_hash_id`, `content_hash_id`, `date`, `access_profile`.

* **Excluded Fields & Reason:**
  * `conversions`: Excluded due to inconsistent tracking setup across clients.
  * `clicks_next_30d`: Excluded from model features as it represents future ground truth (target leakage).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [29]:
import duckdb
from google.colab import userdata

# Retrieve HF_TOKEN securely from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Initialize DuckDB
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# UPDATED: Use DuckDB's modern Secrets Manager to authenticate HTTP requests
con.execute(f"""
CREATE SECRET hf_auth (
    TYPE HTTP,
    BEARER_TOKEN '{hf_token}'
);
""")

In [53]:
# Initialize DuckDB
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# Create HTTP secret for Hugging Face authentication
con.execute(f"""
CREATE SECRET hf_auth (
    TYPE HTTP,
    BEARER_TOKEN '{hf_token}'
);
""")

In [54]:
# NEW (CORRECT SYNTAX):
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# Use DuckDB Secret Manager for HTTP Bearer Authentication
con.execute(f"""
CREATE SECRET hf_auth (
    TYPE HTTP,
    BEARER_TOKEN '{hf_token}'
);
""")

In [20]:
df_clients.isnull().sum()

,0
client_hash_id,0
is_active,10
has_gsc_access,10
has_ga4_access,10
access_profile,0
client_created_date,10
client_updated_date,10
gsc_data_start,37
ga4_data_start,53


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Soln:**

* **Unobserved GA4 Analytics & Off-Page External Factors:**
  A notable portion of active clients in `dim_clients` only have Google Search Console (`gsc_only`) access configured without GA4 tracking (`has_ga4_access = False`). Consequently, user engagement signals (such as dwell time or bounce rates) and external off-page SEO factors (backlink acquisition, seasonal keyword spikes) are unobserved in this slice, creating a potential blind spot for predicting conversion intent.

## Self-check

Before you submit, confirm each line honestly:

- [Done] Every section above is filled — markdown thinking AND the code that backs it
- [Dpne] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Done] No client names, URLs, or private queries anywhere
- [Done] My claims use careful words: observed, measured, directional, decision-support
- [Done] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.